# Proyecto Final - Modelos - Capital Analyst
**Econometría Financiera (FZ4038)** · Bárbara Martínez Flores (A00919623) · Noviembre 2026

Este notebook contiene el mismo código y las mismas interpretaciones que el archivo `ProyectoFinal_BMF.Rmd`/`.html`.

**Cómo correrlo en Google Colab:**
1. Este notebook usa **R**: en Colab, entra a *Entorno de ejecución → Cambiar tipo de entorno de ejecución* y elige **R**.
2. En el panel de archivos (ícono de carpeta, a la izquierda), crea una carpeta `datos` y sube ahí los 24 archivos de Excel de LSEG. Sube las imágenes `.png` a la carpeta principal (`/content`).
3. Corre la primera celda para instalar los paquetes (tarda unos minutos) y después *Entorno de ejecución → Ejecutar todas*.



# Introducción

El objetivo de este proyecto es aplicar las herramientas vistas en el curso a las acciones del portafolio que administro en HowTheMarketWorks (concurso *Econometría Financiera EGADE Oct 26*). El portafolio tiene **11 acciones**: NVIDIA (NVDA), Amazon (AMZN), Caterpillar (CAT), Walmart (WMT), Costco (COST), Synaptics (SYNA), Penguin Solutions (PENG), Freeport-McMoRan (FCX), Apple (AAPL), Exxon Mobil (XOM) y una posición en corto de Goldman Sachs (GS). Con ellas se estiman:

| Sección | Modelo | Datos |
|---|---|---|
| 2 | Regresión lineal simple (beta contra el S&P 500) | Precios diarios de LSEG, 2 ene 2025 – 6 nov 2026 |
| 3 | Matriz de correlación | Precios diarios de LSEG |
| 4 | Regresión lineal múltiple (diaria e intradía de 30 minutos) | LSEG diario y de 30 minutos; comparación con Bloomberg |
| 5 | Datos de panel | Reportes trimestrales de LSEG (10 empresas, 2019T3 – 2026T2) |
| 6 | Series de tiempo y pronóstico (ARIMA) | WMT y NVDA, diario e intradía |
| 7 | GARCH (volatilidad) | WMT y NVDA, diario 2016 – 2026 |
| 8 | Estrategia en HowTheMarketWorks | Resultados de las secciones anteriores |

Todos los precios provienen de LSEG Workspace (*Price History*, precio de cierre ajustado por *splits*) y los estados financieros de LSEG (*Financial Summary*, vista estandarizada, millones de dólares). Los modelos se comparan con los resultados obtenidos en Bloomberg en las actividades anteriores.

# 1. Librerías y datos


In [ ]:
install.packages("readxl")
install.packages("xts")
install.packages("plm")
install.packages("lmtest")
install.packages("sandwich")
install.packages("car")
install.packages("forecast")
install.packages("urca")
install.packages("fGarch")
install.packages("corrplot")
install.packages("IRdisplay")



Las instalaciones se corren una sola vez en la consola, por eso este chunk no se ejecuta al tejer el documento.


In [ ]:
library(readxl)
library(xts)
library(plm)
library(lmtest)
library(sandwich)
library(car)
library(forecast)
library(urca)
library(fGarch)
library(corrplot)


In [ ]:
carpeta <- "datos/"

# Lee un archivo de "Price History" de LSEG (diario o intradía) y regresa una serie xts con el cierre.
# LSEG guarda la fecha y hora como número de Excel en hora universal (UTC).
leer_precios <- function(archivo) {
  crudo <- suppressMessages(read_excel(paste0(carpeta, archivo), col_names = FALSE, col_types = "text"))
  fila  <- which(crudo[[1]] == "Exchange Date")[1]
  datos <- suppressMessages(read_excel(paste0(carpeta, archivo), skip = fila - 1))
  datos <- datos[!is.na(datos$Close) & !is.na(datos$`Exchange Date`), ]
  fecha <- as.POSIXct(round(as.numeric(datos$`Exchange Date`) * 86400), origin = "1899-12-30", tz = "UTC")
  xts(datos$Close, order.by = fecha)
}

# Muestra una imagen solo si el archivo existe en la carpeta del proyecto.
imagen <- function(archivo) {
  if (file.exists(archivo)) IRdisplay::display_png(file = archivo) else cat("Imagen no encontrada:", archivo)
}

# Muestra las tablas de knitr como HTML dentro del notebook
tabla_html <- function(...) IRdisplay::display_html(as.character(knitr::kable(..., format = "html")))
options(scipen = 999, knitr.kable.NA = "–")


In [ ]:
archivos_diarios <- c(NVDA = "NVDA_Price_History_20261107_1631.xlsx",
                      AMZN = "AMZN_Price_History_20261107_1633.xlsx",
                      CAT  = "CAT_Price_History_20261107_1633.xlsx",
                      WMT  = "WMT_Price_History_20261107_1634.xlsx",
                      COST = "COST_Price_History_20261107_1634.xlsx",
                      SYNA = "SYNA_Price_History_20261107_1635.xlsx",
                      PENG = "PENG_Price_History_20261107_1636.xlsx",
                      FCX  = "FCX_Price_History_20261107_1637.xlsx",
                      AAPL = "AAPL_Price_History_20261107_1638.xlsx",
                      XOM  = "XOM_Price_History_20261107_1639.xlsx",
                      GS   = "GS_Price_History_20261107_1639.xlsx",
                      SPX  = "SPX_Price_History_20261107_1639.xlsx")

precios <- do.call(merge, lapply(archivos_diarios, leer_precios))
colnames(precios) <- names(archivos_diarios)
index(precios) <- as.Date(index(precios))

nvidia_returns_daily      <- 100 * (precios$NVDA / lag(precios$NVDA) - 1)
amazon_returns_daily      <- 100 * (precios$AMZN / lag(precios$AMZN) - 1)
caterpillar_returns_daily <- 100 * (precios$CAT  / lag(precios$CAT)  - 1)
walmart_returns_daily     <- 100 * (precios$WMT  / lag(precios$WMT)  - 1)
costco_returns_daily      <- 100 * (precios$COST / lag(precios$COST) - 1)
synaptics_returns_daily   <- 100 * (precios$SYNA / lag(precios$SYNA) - 1)
penguin_returns_daily     <- 100 * (precios$PENG / lag(precios$PENG) - 1)
freeport_returns_daily    <- 100 * (precios$FCX  / lag(precios$FCX)  - 1)
apple_returns_daily       <- 100 * (precios$AAPL / lag(precios$AAPL) - 1)
exxon_returns_daily       <- 100 * (precios$XOM  / lag(precios$XOM)  - 1)
goldman_returns_daily     <- 100 * (precios$GS   / lag(precios$GS)   - 1)
sp500_returns_daily       <- 100 * (precios$SPX  / lag(precios$SPX)  - 1)

rendimientos <- merge(nvidia_returns_daily, amazon_returns_daily, caterpillar_returns_daily,
                      walmart_returns_daily, costco_returns_daily, synaptics_returns_daily,
                      penguin_returns_daily, freeport_returns_daily, apple_returns_daily,
                      exxon_returns_daily, goldman_returns_daily, sp500_returns_daily)
colnames(rendimientos) <- names(archivos_diarios)

rend_2025 <- na.omit(rendimientos["2025-01-03/2026-11-06"])
nrow(precios)
nrow(rend_2025)
tail(precios, 3)



**Interpretación.** Se leyeron los 12 archivos diarios de LSEG (11 acciones y el S&P 500), con 2728 días de cotización del 4 de enero de 2016 al 6 de noviembre de 2026. Los rendimientos diarios se calculan en porcentaje, igual que en Bloomberg (*Transformation: Percent*). Para las regresiones simples y la matriz de correlación se usa el periodo del 2 de enero de 2025 al 6 de noviembre de 2026, con 463 rendimientos diarios: es el mismo periodo de las actividades anteriores, actualizado hasta el último dato disponible.

# 2. Regresión lineal simple: beta de cada acción contra el S&P 500


In [ ]:
acciones <- c("NVDA","AMZN","CAT","WMT","COST","SYNA","PENG","FCX","AAPL","XOM","GS")
modelos_simples <- lapply(acciones, function(a) lm(rend_2025[, a] ~ rend_2025[, "SPX"]))
names(modelos_simples) <- acciones

betas <- data.frame(
  Accion  = acciones,
  Alfa    = sapply(modelos_simples, function(m) coef(m)[1]),
  Beta    = sapply(modelos_simples, function(m) coef(m)[2]),
  t_Beta  = sapply(modelos_simples, function(m) summary(m)$coefficients[2, 3]),
  R2      = sapply(modelos_simples, function(m) summary(m)$r.squared))
betas <- betas[order(-betas$Beta), ]
tabla_html(betas, digits = 3, row.names = FALSE,
             col.names = c("Acción", "Alfa (%)", "Beta", "t de la beta", "R²"),
             caption = "Regresión simple de cada acción contra el S&P 500 (2 ene 2025 – 6 nov 2026)")



**Interpretación.** La beta indica cuánto se mueve en promedio cada acción cuando el S&P 500 se mueve 1%. Todas las betas son significativas al 5% (t mayor a 1.96); la de XOM apenas lo es (t de 2.04).

- **Acciones agresivas (beta mayor a 1):** PENG (2.51), SYNA (2.09), NVDA (1.95), FCX (1.83), GS (1.48), AMZN (1.37), CAT (1.37) y AAPL (1.08). Por ejemplo, si el S&P 500 cae 1%, PENG cae en promedio 2.51%.
- **Acciones defensivas (beta menor a 1):** WMT (0.39), COST (0.25) y XOM (0.15). Se mueven mucho menos que el mercado.
- **R²:** el mercado explica más de la mitad de la variación diaria de GS (57.4%) y NVDA (52%), pero casi nada de la de WMT (6.3%), COST (3.9%) y XOM (0.9%), que dependen sobre todo de factores propios.
- Los resultados son muy parecidos a los de la Tarea 4, cuyo periodo terminaba el 12 de octubre (por ejemplo, NVDA tenía una beta de 1.947 y GS de 1.471); la pequeña diferencia se debe a que aquí el periodo llega al 6 de noviembre.

## 2.1 Beta del portafolio actual


In [ ]:
imagen("hmw_final_posiciones.png")


In [ ]:
posiciones <- data.frame(
  Accion  = c("NVDA","AMZN","CAT","WMT","COST","SYNA","PENG","FCX","AAPL","XOM","GS (corto)","Efectivo"),
  Titulos = c(685, 571, 184, 1060, 83, 583, 897, 676, 1, 340, -50, NA),
  Valor   = c(127848.40, 148060.30, 102787.92, 133051.20, 87445.48, 68036.10, 37521.51,
              40167.92, 342.37, 56395.80, -43291.00, 83400.94 + 44050.00))
posiciones$Peso <- posiciones$Valor / sum(posiciones$Valor) * 100
posiciones$Beta <- c(sapply(acciones[acciones != "GS"], function(a) unname(coef(modelos_simples[[a]])[2])),
                     unname(coef(modelos_simples[["GS"]])[2]), 0)
posiciones$Contribucion <- posiciones$Peso / 100 * posiciones$Beta
beta_portafolio <- sum(posiciones$Contribucion)
tabla_html(posiciones, digits = c(0, 0, 2, 2, 3, 3), row.names = FALSE,
             format.args = list(big.mark = ","),
             col.names = c("Activo", "Títulos", "Valor (USD)", "Peso (%)", "Beta", "Peso × Beta"),
             caption = "Portafolio en HowTheMarketWorks al 6 de noviembre de 2026")



**Interpretación.** Con las posiciones al cierre del 6 de noviembre (la posición corta de GS con valor negativo y el efectivo incluyendo los $44,050 recibidos por el corto), la suma de la tabla es de $885,816.94, igual al *Portfolio Value* que reporta el simulador ($885,816.94). La **beta del portafolio es de 1.04**: si el S&P 500 cae 1%, el portafolio cae en promedio alrededor de 1.04%. Las posiciones que más riesgo de mercado aportan son AMZN, NVDA y CAT, mientras que la posición corta en GS y el efectivo (14.4% del portafolio) la reducen.

# 3. Matriz de correlación


In [ ]:
matriz_cor <- cor(rend_2025)
tabla_html(round(matriz_cor, 2), caption = "Matriz de correlación de los rendimientos diarios (2 ene 2025 – 6 nov 2026)")
corrplot(matriz_cor, method = "color", type = "upper", addCoef.col = "black", number.cex = 0.7,
         tl.col = "black", tl.srt = 45, col = colorRampPalette(c("#B2182B", "white", "#2166AC"))(200))



**Interpretación.**

- Las correlaciones más altas con el S&P 500 son las de GS (0.76) y NVDA (0.72), que son las acciones más ligadas al mercado.
- La correlación más alta entre dos acciones del portafolio es la de **WMT y COST (0.67)**: ambas son tiendas de autoservicio y responden a las mismas condiciones del consumo, como se vio en el modelo de Retail de la Actividad 3.
- Las acciones de tecnología y ciclo económico (NVDA, SYNA, PENG, CAT, FCX y GS) tienen correlaciones de entre 0.4 y 0.6 entre sí: tienden a caer juntas en los días malos.
- **XOM** prácticamente no está correlacionada con ninguna (todas sus correlaciones son menores a 0.15), y **WMT y COST** tienen correlaciones casi nulas con NVDA, PENG y CAT. Por eso son las que más diversifican el portafolio.

# 4. Regresión lineal múltiple

## 4.1 Modelo de Apple contra Bloomberg (dos periodos)

Se replica el modelo de la Tarea 5, **AAPL ~ CAT + GS + FCX + XOM + COST + AMZN + WMT + NVDA**, ahora con los precios de LSEG, y se compara con los resultados de Bloomberg (*Multi-Security Regression Analysis*).


In [ ]:
imagen("bbg_mra_aapl_2025.png")


In [ ]:
formula_aapl <- AAPL ~ CAT + GS + FCX + XOM + COST + AMZN + WMT + NVDA
datos_aapl_2025 <- as.data.frame(na.omit(rendimientos["2025-01-03/2026-10-12"]))
datos_aapl_2018 <- as.data.frame(na.omit(rendimientos["2018-01-03/2023-06-30",
                                   c("AAPL","CAT","GS","FCX","XOM","COST","AMZN","WMT","NVDA")]))
modelo_aapl_2025 <- lm(formula_aapl, data = datos_aapl_2025)
modelo_aapl_2018 <- lm(formula_aapl, data = datos_aapl_2018)
summary(modelo_aapl_2025)
vif(modelo_aapl_2025)


In [ ]:
comp_aapl <- data.frame(
  Variable  = c("Intercepto","CAT","GS","FCX","XOM","COST","AMZN","WMT","NVDA"),
  BBG_2025  = c(0.0102, -0.0384, 0.2358, 0.0429, 0.1381, 0.1485, 0.1489, 0.1691, 0.0557),
  R_2025    = coef(modelo_aapl_2025),
  BBG_2018  = c(0.0418, -0.0084, 0.1623, 0.0155, 0.0585, 0.2432, 0.2616, 0.0463, 0.1657),
  R_2018    = coef(modelo_aapl_2018))
tabla_html(comp_aapl, digits = 4, row.names = FALSE,
             col.names = c("Variable", "Bloomberg 2025–26", "RStudio 2025–26", "Bloomberg 2018–23", "RStudio 2018–23"),
             caption = "Modelo de AAPL: coeficientes en Bloomberg vs. RStudio (datos de LSEG)")
resumen_aapl <- data.frame(
  Estadistico = c("R² (%)", "Observaciones"),
  BBG_2025 = c(29.5, 438), R_2025 = c(100 * summary(modelo_aapl_2025)$r.squared, nobs(modelo_aapl_2025)),
  BBG_2018 = c(58.1, 1360), R_2018 = c(100 * summary(modelo_aapl_2018)$r.squared, nobs(modelo_aapl_2018)))
tabla_html(resumen_aapl, digits = 1, row.names = FALSE,
             col.names = c("Estadístico", "Bloomberg 2025–26", "RStudio 2025–26", "Bloomberg 2018–23", "RStudio 2018–23"))


In [ ]:
imagen("bbg_mra_aapl_2018.png")



**Interpretación.** Con los precios de LSEG se obtienen los mismos coeficientes que en la Tarea 5 con Yahoo Finance para 2025–2026 y prácticamente los mismos para 2018–2023, y siguen siendo muy cercanos a los de Bloomberg: la mayor diferencia es la de COST en 2025–2026 (0.1485 contra 0.1637). Las diferencias se explican por el número de observaciones (444 en RStudio contra 438 en Bloomberg). En el periodo reciente el modelo explica el 29.9% de la variación de Apple, contra el 58.6% de 2018–2023: Apple depende hoy menos de estas acciones. En 2025–2026, por cada 1% que sube GS, Apple sube 0.23%, y por cada 1% de AMZN, 0.15%, manteniendo lo demás constante. Los VIF son todos menores a 2.1, por lo que no hay multicolinealidad.

## 4.2 Modelo intradía (30 minutos): Walmart contra Costco, Amazon y el S&P 500


In [ ]:
archivos_intradia <- c(WMT  = "WMT_30_Price_History_20261107_1646.xlsx",
                       NVDA = "NVDA_30_Price_History_20261107_1649.xlsx",
                       COST = "COST_30_Price_History_20261107_1650.xlsx",
                       AMZN = "AMZN_30_Price_History_20261107_1651.xlsx",
                       SPX  = "SPX_30Price_History_20261107_1652.xlsx")
intradia <- do.call(merge, lapply(archivos_intradia, leer_precios))
colnames(intradia) <- names(archivos_intradia)

# Hora de Nueva York y solo horario regular: barras que cierran de las 10:00 a las 16:00 (13 por día)
tzone(intradia) <- "America/New_York"
hora <- format(index(intradia), "%H:%M")
intradia <- na.omit(intradia[hora >= "10:00" & hora <= "16:00"])

# Rendimientos logarítmicos de 30 minutos, sin el salto de la noche (primera barra de cada día)
rend_intradia <- 100 * diff(log(intradia))
dia <- as.Date(index(rend_intradia), tz = "America/New_York")
primera_barra <- c(TRUE, dia[-1] != dia[-length(dia)])
rend_intradia <- na.omit(rend_intradia[!primera_barra])

nrow(intradia)
length(unique(as.Date(index(intradia), tz = "America/New_York")))
nrow(rend_intradia)



**Interpretación.** Los archivos de 30 minutos de LSEG incluyen el horario extendido (de las 4:30 a las 20:00). Se conservan solo las 13 barras del horario regular (de 9:30 a 16:00, hora de Nueva York) de 69 días de mercado, del 3 de agosto al 6 de noviembre de 2026. Para que los rendimientos sean realmente de 30 minutos, se elimina el rendimiento de la primera barra de cada día, porque incluye el salto de precio de la noche anterior. Quedan 828 rendimientos.


In [ ]:
modelo_intradia <- lm(WMT ~ COST + AMZN + SPX, data = as.data.frame(rend_intradia))
summary(modelo_intradia)
vif(modelo_intradia)
dwtest(modelo_intradia)



**Interpretación.** En intervalos de 30 minutos, el modelo explica el 46.5% de los movimientos de Walmart. Por cada 1% que sube Costco en media hora, Walmart sube en promedio 0.76%; y por cada 1% del S&P 500, 0.14%. Amazon tiene un coeficiente pequeño y negativo (-0.05, valor p de 0.034): una vez que se toma en cuenta el movimiento del mercado, Amazon prácticamente no aporta información sobre Walmart. La relación con Costco confirma, ahora dentro del día, el vínculo que se vio en los datos diarios. Los VIF son bajos y la prueba de Durbin-Watson (d = 1.97, valor p de 0.340) no muestra autocorrelación, por lo que los errores estándar son confiables.

# 5. Datos de panel: flujo de efectivo de operación de las empresas del portafolio

Siguiendo la metodología de la Tarea 6, se estudia qué factores explican el **flujo de efectivo de operación** de 10 empresas del portafolio (se excluye GS porque es un banco y sus estados financieros no son comparables). Los datos son trimestrales, del tercer trimestre de 2019 al segundo de 2026, en millones de dólares.


In [ ]:
leer_hoja <- function(archivo, hoja_nombre, campos_buscar) {
  hoja <- suppressMessages(read_excel(paste0(carpeta, archivo), sheet = hoja_nombre,
                                      col_names = FALSE, col_types = "text"))
  campos <- hoja[[1]]
  fechas <- as.Date(as.numeric(unlist(hoja[which(campos == "Period End Date")[1], -1])), origin = "1899-12-30")
  datos <- data.frame(fecha = fechas)
  for (n in names(campos_buscar)) {
    datos[[n]] <- as.numeric(unlist(hoja[which(campos == campos_buscar[n])[1], -1]))
  }
  datos[!is.na(datos$fecha), ]
}

leer_lseg <- function(archivo, empresa) {
  resumen <- leer_hoja(archivo, "Financial Summary",
                       c(flujo_op = "Net Cash Flow from Operating Activities",
                         ventas   = "Revenue from Business Activities - Total",
                         deuda    = "Debt - Total"))
  balance <- leer_hoja(archivo, "Balance Sheet", c(inventarios = "Inventories - Total"))
  datos <- merge(resumen, balance, by = "fecha")
  datos$empresa <- empresa
  datos
}

panel <- rbind(leer_lseg("NVDA_CF-Export-07-11-2026.xlsx", "NVDA"),
               leer_lseg("AMZN_CF-Export-01-11-2026.xlsx", "AMZN"),
               leer_lseg("CAT_CF-Export-07-11-2026.xlsx",  "CAT"),
               leer_lseg("WMT_CF-Export-01-11-2026.xlsx",  "WMT"),
               leer_lseg("COST_CF-Export-01-11-2026.xlsx", "COST"),
               leer_lseg("SYNA_CF-Export-07-11-2026.xlsx", "SYNA"),
               leer_lseg("PENG_CF-Export-07-11-2026.xlsx", "PENG"),
               leer_lseg("FCX_CF-Export-07-11-2026.xlsx",  "FCX"),
               leer_lseg("AAPL_CF-Export-07-11-2026.xlsx", "AAPL"),
               leer_lseg("XOM_CF-Export-07-11-2026.xlsx",  "XOM"))

# Trimestre calendario: se resta 10 días a la fecha de cierre para que un trimestre fiscal que
# termina en los primeros días del mes (por ejemplo, Apple el 1 de julio) quede en el trimestre correcto.
fecha_ajustada <- panel$fecha - 10
panel$trimestre <- paste0(format(fecha_ajustada, "%Y"), "T",
                          (as.numeric(format(fecha_ajustada, "%m")) - 1) %/% 3 + 1)
panel <- panel[panel$trimestre >= "2019T3" & panel$trimestre <= "2026T2",
               c("empresa", "trimestre", "fecha", "flujo_op", "ventas", "inventarios", "deuda")]
panel <- panel[order(panel$empresa, panel$trimestre), ]
rownames(panel) <- NULL

table(panel$empresa)
any(duplicated(panel[, c("empresa", "trimestre")]))
aggregate(cbind(flujo_op, ventas, inventarios, deuda) ~ empresa, data = panel, FUN = mean)



**Interpretación.** Se forma un **panel balanceado** de 10 empresas observadas durante 28 trimestres (280 observaciones), sin trimestres repetidos ni datos faltantes. Como varias empresas no cierran su año fiscal en diciembre, cada reporte se asigna al trimestre calendario en el que termina. Las empresas son muy distintas en tamaño: Apple genera en promedio más de 28,000 millones de dólares de flujo de operación por trimestre, mientras que Penguin Solutions y Synaptics generan menos de 70 millones. Esas diferencias propias de cada empresa son las que controlan los modelos de panel.


In [ ]:
formula_panel <- flujo_op ~ ventas + inventarios + deuda
panel_po <- plm(formula_panel, data = panel, model = "pooling", index = c("empresa", "trimestre"))
summary(panel_po)
pFtest(formula_panel, data = panel, model = "within", index = c("empresa", "trimestre"))


In [ ]:
panel_fe <- plm(formula_panel, data = panel, model = "within", index = c("empresa", "trimestre"))
summary(panel_fe)
panel_re <- plm(formula_panel, data = panel, model = "random", index = c("empresa", "trimestre"))
summary(panel_re)
phtest(panel_fe, panel_re)


In [ ]:
pwartest(formula_panel, data = panel, index = c("empresa", "trimestre"))
pcdtest(formula_panel, data = panel, index = c("empresa", "trimestre"))
coeftest(panel_fe, vcov = vcovSCC(panel_fe, type = "HC1"))


In [ ]:
p_f   <- pFtest(formula_panel, data = panel, model = "within", index = c("empresa", "trimestre"))
p_h   <- phtest(panel_fe, panel_re)
p_w   <- pwartest(formula_panel, data = panel, index = c("empresa", "trimestre"))
p_cd  <- pcdtest(formula_panel, data = panel, index = c("empresa", "trimestre"))
cf_fe <- coef(summary(panel_fe))



**Interpretación.**

- **Modelo agrupado (pooled):** sin distinguir entre empresas, las tres variables son significativas y explican el 74.7% de la variación del flujo de operación, pero la deuda aparece con signo positivo (0.048) porque las empresas más grandes tienen a la vez más deuda y más flujo. Al controlar por empresa, ese signo cambia, lo que muestra que el modelo agrupado mezcla las diferencias de tamaño con la relación entre las variables.
- **Pooled vs. efectos fijos:** la prueba F de efectos individuales (F = 37.99, valor p menor a 0.001) indica que cada empresa tiene un intercepto propio, por lo que el modelo agrupado no es adecuado.
- **Efectos fijos vs. aleatorios:** la prueba de Hausman (ji cuadrada = 44.88, valor p menor a 0.001) rechaza que las diferencias entre empresas sean independientes de las variables, por lo que se elige el **modelo de efectos fijos**. Además, las 10 empresas no son una muestra aleatoria: son las de mi portafolio.
- **Resultados del modelo de efectos fijos:** dentro de cada empresa, cuando las ventas del trimestre suben un millón de dólares, el flujo de operación sube en promedio 0.40 millones; cuando los inventarios suben un millón, el flujo baja 0.40 millones, porque el efectivo queda invertido en mercancía; y cuando la deuda sube un millón, el flujo baja 0.07 millones (valor p de 0.007). Las tres variables explican el 69.7% de la variación del flujo de operación dentro de cada empresa.
- **Validez de los errores estándar:** la prueba de Wooldridge (valor p de 0.257) no encuentra autocorrelación, y la de Pesaran (valor p de 0.312) no encuentra dependencia entre empresas. Aun así, con los errores estándar de Driscoll-Kraay las tres variables siguen siendo significativas al 5%.
- **Comparación con la Tarea 6:** con las 5 empresas de autoservicio, el efecto de las ventas fue de 0.39 y el de los inventarios de −0.71. Con las 10 empresas del portafolio, el efecto de las ventas es casi el mismo, y el de los inventarios es menor, porque en tecnología y energía los inventarios pesan menos que en el comercio minorista.

# 6. Series de tiempo y pronóstico: WMT y NVDA

Siguiendo el capítulo 21 de Gujarati, primero se revisa si los precios son estacionarios (gráfica, correlograma y prueba Dickey-Fuller), después se elige un modelo ARIMA y finalmente se pronostica. Se trabaja con el **logaritmo del precio**, cuya primera diferencia es el rendimiento logarítmico, como en la pestaña de ruido blanco de Walmart.

## 6.1 Estacionariedad (datos diarios, 2 ene 2025 – 6 nov 2026)


In [ ]:
log_wmt  <- log(precios["2025-01-02/2026-11-06", "WMT"])
log_nvda <- log(precios["2025-01-02/2026-11-06", "NVDA"])
par(mfrow = c(2, 2))
plot(as.numeric(log_wmt), type = "l", main = "Logaritmo del precio de WMT", xlab = "Día", ylab = "ln(P)")
plot(100 * diff(as.numeric(log_wmt)), type = "l", main = "Rendimiento logarítmico de WMT (%)", xlab = "Día", ylab = "%")
plot(as.numeric(log_nvda), type = "l", main = "Logaritmo del precio de NVDA", xlab = "Día", ylab = "ln(P)")
plot(100 * diff(as.numeric(log_nvda)), type = "l", main = "Rendimiento logarítmico de NVDA (%)", xlab = "Día", ylab = "%")
par(mfrow = c(1, 1))


In [ ]:
par(mfrow = c(2, 2))
acf(as.numeric(log_wmt), lag.max = 36, main = "FAC: ln(precio) WMT")
acf(diff(as.numeric(log_wmt)), lag.max = 36, main = "FAC: rendimiento WMT")
acf(as.numeric(log_nvda), lag.max = 36, main = "FAC: ln(precio) NVDA")
acf(diff(as.numeric(log_nvda)), lag.max = 36, main = "FAC: rendimiento NVDA")
par(mfrow = c(1, 1))


In [ ]:
prueba_df <- function(serie, nombre) {
  tipos <- c(none = "Sin intercepto ni tendencia", drift = "Con intercepto", trend = "Con intercepto y tendencia")
  do.call(rbind, lapply(names(tipos), function(tp) {
    u <- ur.df(as.numeric(serie), type = tp, lags = 0)
    data.frame(Serie = nombre, Especificacion = tipos[tp], tau = u@teststat[1],
               Critico_5 = u@cval[1, "5pct"], Conclusion = ifelse(u@teststat[1] < u@cval[1, "5pct"],
                                                               "Estacionaria", "Raíz unitaria"))
  }))
}
tabla_df <- rbind(prueba_df(log_wmt, "ln(precio) WMT"),
                  prueba_df(100 * diff(as.numeric(log_wmt)), "Rendimiento WMT"),
                  prueba_df(log_nvda, "ln(precio) NVDA"),
                  prueba_df(100 * diff(as.numeric(log_nvda)), "Rendimiento NVDA"))
tabla_html(tabla_df, digits = 3, row.names = FALSE,
             col.names = c("Serie", "Especificación", "τ calculada", "τ crítica (5%)", "Conclusión"),
             caption = "Prueba Dickey-Fuller")
# Dickey-Fuller aumentada con rezagos elegidos por el criterio de Akaike
summary(ur.df(as.numeric(log_nvda), type = "trend", lags = 10, selectlags = "AIC"))@teststat
summary(ur.df(as.numeric(log_wmt),  type = "trend", lags = 10, selectlags = "AIC"))@teststat



**Interpretación.**

- **Gráficas:** el logaritmo del precio de WMT tiene una tendencia creciente y el de NVDA sube y baja sin regresar a un nivel fijo. Los rendimientos, en cambio, se mueven alrededor de cero con una amplitud más o menos constante, como el ruido blanco.
- **Correlogramas:** la FAC del logaritmo del precio empieza cerca de 1 y baja muy lentamente, que es el patrón de una serie no estacionaria (figura 21.7 de Gujarati). La FAC de los rendimientos cae de inmediato y casi todas sus barras quedan dentro de las bandas de confianza, que es el patrón de una serie estacionaria.
- **Dickey-Fuller:** para el logaritmo del precio, en las tres especificaciones la τ calculada es menos negativa que la τ crítica al 5%, por lo que **no se rechaza la raíz unitaria**: los precios siguen una caminata aleatoria. Lo mismo ocurre con la prueba aumentada (DFA) con rezagos elegidos por Akaike. Para los rendimientos, la τ es muy negativa (entre −20 y −24) y se rechaza la raíz unitaria: **los rendimientos son estacionarios**.
- **Conclusión:** el logaritmo del precio de WMT y de NVDA es **I(1)**, un proceso estacionario en diferencias (PED). Por eso el modelo debe trabajarse en primeras diferencias (d = 1), y no basta con quitarle una tendencia lineal.

## 6.2 Modelo ARIMA y pronóstico diario (10 días hábiles)


In [ ]:
arima_wmt  <- auto.arima(as.numeric(log_wmt),  stepwise = FALSE, approximation = FALSE)
arima_nvda <- auto.arima(as.numeric(log_nvda), stepwise = FALSE, approximation = FALSE)
arima_wmt
arima_nvda
Box.test(residuals(arima_wmt),  lag = 10, type = "Ljung-Box", fitdf = sum(arimaorder(arima_wmt)[c(1, 3)]))
Box.test(residuals(arima_nvda), lag = 10, type = "Ljung-Box", fitdf = sum(arimaorder(arima_nvda)[c(1, 3)]))


In [ ]:
pron_wmt  <- forecast(arima_wmt,  h = 10)
pron_nvda <- forecast(arima_nvda, h = 10)
tabla_pron <- data.frame(
  Dia = 1:10,
  WMT = exp(pron_wmt$mean), WMT_inf = exp(pron_wmt$lower[, 2]), WMT_sup = exp(pron_wmt$upper[, 2]),
  NVDA = exp(pron_nvda$mean), NVDA_inf = exp(pron_nvda$lower[, 2]), NVDA_sup = exp(pron_nvda$upper[, 2]))
tabla_html(tabla_pron, digits = 2, row.names = FALSE,
             col.names = c("Día hábil", "WMT", "WMT mín. 95%", "WMT máx. 95%", "NVDA", "NVDA mín. 95%", "NVDA máx. 95%"),
             caption = "Pronóstico del precio de cierre (USD) a partir del 6 de noviembre de 2026")
par(mfrow = c(1, 2))
plot(pron_wmt,  include = 120, main = "WMT: pronóstico ARIMA (ln del precio)")
plot(pron_nvda, include = 120, main = "NVDA: pronóstico ARIMA (ln del precio)")
par(mfrow = c(1, 1))



**Interpretación.**

- **WMT:** el modelo elegido por el criterio de Akaike es un **ARIMA(0,1,0)**, es decir, una caminata aleatoria pura: la mejor predicción del precio de mañana es el precio de hoy, como describe Gujarati para los precios de las acciones (ecuación 21.3.4). Por eso el pronóstico es plano en $125.52, y lo único que cambia con el horizonte es la incertidumbre: a 10 días, el precio estaría con 95% de confianza entre $113.32 y $139.03.
- **NVDA:** el modelo elegido es un **ARIMA(0,1,4)**: los rendimientos tienen cierta memoria de corto plazo (componentes de promedio móvil), en parte por la fuerte caída del 5 de noviembre (−9.3%). El pronóstico sugiere un rebote moderado a unos $189.86 en 10 días, pero el intervalo de 95% es muy amplio (de $164.07 a $219.70), así que el rebote no es estadísticamente distinto de quedarse en el precio actual.
- **Diagnóstico:** la prueba de Ljung-Box sobre los residuos tiene valores p mayores a 0.05 en ambos modelos, por lo que los residuos son ruido blanco y los modelos capturan la estructura de la serie.

## 6.3 Pronóstico intradía (13 barras de 30 minutos = un día de mercado)


In [ ]:
log_wmt_30  <- log(as.numeric(intradia$WMT))
log_nvda_30 <- log(as.numeric(intradia$NVDA))
arima_wmt_30  <- auto.arima(log_wmt_30)
arima_nvda_30 <- auto.arima(log_nvda_30)
arima_wmt_30
arima_nvda_30
pron_wmt_30  <- forecast(arima_wmt_30,  h = 13)
pron_nvda_30 <- forecast(arima_nvda_30, h = 13)
tabla_30 <- data.frame(
  Barra = c("10:00", "16:00"),
  WMT = exp(pron_wmt_30$mean[c(1, 13)]), WMT_inf = exp(pron_wmt_30$lower[c(1, 13), 2]), WMT_sup = exp(pron_wmt_30$upper[c(1, 13), 2]),
  NVDA = exp(pron_nvda_30$mean[c(1, 13)]), NVDA_inf = exp(pron_nvda_30$lower[c(1, 13), 2]), NVDA_sup = exp(pron_nvda_30$upper[c(1, 13), 2]))
tabla_html(tabla_30, digits = 2, row.names = FALSE,
             col.names = c("Barra del día siguiente", "WMT", "WMT mín. 95%", "WMT máx. 95%", "NVDA", "NVDA mín. 95%", "NVDA máx. 95%"),
             caption = "Pronóstico intradía del precio (USD) para el siguiente día de mercado")
par(mfrow = c(1, 2))
plot(pron_wmt_30,  include = 130, main = "WMT: pronóstico de 30 minutos")
plot(pron_nvda_30, include = 130, main = "NVDA: pronóstico de 30 minutos")
par(mfrow = c(1, 1))



**Interpretación.** Con datos de 30 minutos (897 barras del horario regular), los modelos elegidos son un ARIMA(0,1,0) para WMT y un ARIMA(1,1,0) para NVDA: también dentro del día los precios se comportan prácticamente como una caminata aleatoria. Para el siguiente día de mercado, el precio esperado de WMT es de $125.47, con un rango de 95% al cierre de $121.07 a $130.03; el de NVDA es de $186.73, con un rango de $178.25 a $195.62. El rango de NVDA equivale al 9.3% de su precio, contra 7.1% en WMT: NVDA es más riesgosa, lo que anticipa el resultado de volatilidad de la siguiente sección. Nota: el modelo trata las barras como consecutivas, sin considerar el salto de precio entre el cierre y la apertura del día siguiente.

# 7. GARCH (puntos extra): volatilidad de WMT y NVDA

Los rendimientos son estacionarios en su media, pero su varianza no es constante: hay periodos de calma y periodos de mucha volatilidad. El modelo **GARCH(1,1)** permite que la varianza de hoy dependa del choque de ayer (α) y de la varianza de ayer (β). Se estima con los rendimientos logarítmicos diarios de 2016 a 2026 y una distribución t de Student, porque los rendimientos tienen colas pesadas.


In [ ]:
rend_log_wmt  <- 100 * diff(log(as.numeric(precios$WMT)))
rend_log_nvda <- 100 * diff(log(as.numeric(precios$NVDA)))
garch_wmt  <- garchFit(~ garch(1, 1), data = rend_log_wmt,  cond.dist = "std", trace = FALSE)
garch_nvda <- garchFit(~ garch(1, 1), data = rend_log_nvda, cond.dist = "std", trace = FALSE)
round(garch_wmt@fit$matcoef, 4)
round(garch_nvda@fit$matcoef, 4)

vol_wmt  <- predict(garch_wmt,  n.ahead = 10)
vol_nvda <- predict(garch_nvda, n.ahead = 10)
tabla_vol <- data.frame(Dia = 1:10, WMT = vol_wmt$standardDeviation, NVDA = vol_nvda$standardDeviation)
tabla_html(tabla_vol, digits = 2, row.names = FALSE,
             col.names = c("Día hábil", "Volatilidad diaria WMT (%)", "Volatilidad diaria NVDA (%)"),
             caption = "Pronóstico de volatilidad GARCH(1,1)")


In [ ]:
par(mfrow = c(1, 2))
plot(tail(garch_wmt@sigma.t, 500),  type = "l", main = "WMT: volatilidad condicional (últimos 500 días)", ylab = "% diario", xlab = "Día")
plot(tail(garch_nvda@sigma.t, 500), type = "l", main = "NVDA: volatilidad condicional (últimos 500 días)", ylab = "% diario", xlab = "Día")
par(mfrow = c(1, 1))


In [ ]:
valor_wmt  <- posiciones$Valor[posiciones$Accion == "WMT"]
valor_nvda <- posiciones$Valor[posiciones$Accion == "NVDA"]
var_wmt  <- 1.645 * vol_wmt$standardDeviation[1]  / 100 * valor_wmt
var_nvda <- 1.645 * vol_nvda$standardDeviation[1] / 100 * valor_nvda
data.frame(Posicion = c("WMT", "NVDA"), Valor = c(valor_wmt, valor_nvda), VaR_95_1dia = c(var_wmt, var_nvda))



**Interpretación.**

- **Coeficientes:** en ambos modelos α y β son significativos y su suma es cercana a 1 (0.89 para WMT y 0.97 para NVDA): la volatilidad es **persistente**, es decir, un día muy volátil suele ir seguido de otros días volátiles. En NVDA el peso de β (0.88) es mayor, por lo que sus periodos de alta volatilidad duran más.
- **Pronóstico de volatilidad:** para el siguiente día hábil, la volatilidad esperada de WMT es de 1.84% diario y la de NVDA de 3.84%, es decir, NVDA es alrededor de 2.1 veces más volátil. La volatilidad de NVDA está elevada por la caída del 5 de noviembre y el modelo prevé que baje muy lentamente.
- **Valor en riesgo (VaR):** con 95% de confianza, la pérdida máxima esperada en un día es de unos $4,038 en la posición de WMT ($133,051) y de unos $8,074 en la de NVDA ($127,848), aunque las dos posiciones tienen un valor parecido.

# 8. Estrategia en HowTheMarketWorks


In [ ]:
imagen("hmw_final_balance.png")


In [ ]:
imagen("hmw_final_ordenes.png")



Al 6 de noviembre de 2026, el portafolio vale $885,816.94, con un rendimiento acumulado de −11.42% contra −4.90% del SPY. Las mayores pérdidas son PENG (−42.8%), CAT (−31.6%), FCX (−19.1%) y NVDA (−18.4%), justamente las acciones con beta más alta; las ganancias vienen de WMT (+11.3%), COST (+9.7%) y la posición corta en GS (+1.7%).

## 8.1 Estrategia

Con base en los resultados anteriores, la estrategia es reducir el riesgo de mercado del portafolio sin dejar de estar invertido:

- **Vender las 897 acciones de PENG.** Es la acción con la beta más alta del portafolio (2.51): si el S&P 500 cae 1%, PENG cae en promedio 2.5%. Además, el mercado solo explica el 30% de sus movimientos, por lo que tiene mucho riesgo propio, y acumula una pérdida de −42.8%.
- **Usar lo obtenido para comprar XOM.** Su beta es de 0.15 y su correlación con el resto de las acciones del portafolio es prácticamente cero, por lo que es la acción que más diversifica.
- **Mantener NVDA, el resto de las posiciones y la posición corta en GS con su stop de $951.** El pronóstico ARIMA de NVDA no muestra una caída significativa, y el corto de GS reduce la beta del portafolio.

Con estos cambios, la beta del portafolio bajó de 1.04 a alrededor de 0.94; el resultado exacto se presenta en la sección 8.3.

## 8.2 Órdenes y ejecución


In [ ]:
imagen("hmw_final_orden_peng.png")


In [ ]:
imagen("hmw_final_orden_xom.png")


In [ ]:
imagen("hmw_final_ordenes_ejecutadas.png")


In [ ]:
precio_peng <- 43.30
precio_xom  <- 164.87   # Precio de XOM en Transaction History
ordenes_finales <- data.frame(
  Num       = 1:2,
  Operacion = c("Market - Sell", "Limit - Buy (GTC)"),
  Accion    = c("PENG", "XOM"),
  Titulos   = c(897, 233),
  Precio_orden = c(NA, 166.00),
  Precio_ejecucion = c(precio_peng, precio_xom),
  Flujo     = c(897 * precio_peng - 10, -(233 * precio_xom + 10)))
tabla_html(ordenes_finales, digits = 2, row.names = FALSE, format.args = list(big.mark = ","),
             col.names = c("#", "Operación", "Acción", "Títulos", "Precio límite", "Precio de ejecución", "Flujo de efectivo (USD)"),
             caption = "Órdenes ejecutadas en HowTheMarketWorks")



**Interpretación.** La venta de PENG se ejecutó a $43.30 por acción, por encima del precio de $41.83 con el que se planeó la operación, e ingresaron $38,830.10 después de la comisión de $10. Con ese dinero se compraron 233 acciones de XOM con una orden límite a $166.00; se ejecutó a $164.87, por debajo del límite, con un costo de $38,424.71. El efectivo pasa de $83,400.94 a $84,106.25 en *Account Balances*: +$405.39 por estas dos operaciones (incluyendo las dos comisiones) más $299.92 del dividendo de CAT registrado en *Transaction History*.

## 8.3 Portafolio final


In [ ]:
imagen("hmw_final_posiciones_despues.png")


In [ ]:
imagen("hmw_final_balance_despues.png")


In [ ]:
final <- data.frame(
  Accion  = c("NVDA","AMZN","CAT","WMT","COST","SYNA","FCX","AAPL","XOM","GS (corto)","Efectivo"),
  Titulos = c(685, 571, 184, 1060, 83, 583, 676, 1, 573, -50, NA),
  Valor   = c(121001.83, 153125.76, 109230.68, 132897.50, 87900.32, 68417.97, 42260.14,
              349.41, 94600.87, -45314.75, 84106.25 + 44050.00))
final$Peso <- final$Valor / sum(final$Valor) * 100
final$Beta <- c(sapply(c("NVDA","AMZN","CAT","WMT","COST","SYNA","FCX","AAPL","XOM","GS"),
                       function(a) unname(coef(modelos_simples[[a]])[2])), 0)
final$Contribucion <- final$Peso / 100 * final$Beta
beta_final <- sum(final$Contribucion)
tabla_html(final, digits = c(0, 0, 2, 2, 3, 3), row.names = FALSE, format.args = list(big.mark = ","),
             col.names = c("Activo", "Títulos", "Valor (USD)", "Peso (%)", "Beta", "Peso × Beta"),
             caption = "Portafolio final en HowTheMarketWorks (9 de noviembre de 2026)")

comparacion <- data.frame(
  Indicador = c("Valor del portafolio (USD)", "Beta del portafolio", "Peso de PENG (%)", "Peso de XOM (%)",
                "Peso del efectivo (%)", "VaR 95% a un día de NVDA (USD)"),
  Antes   = c(sum(posiciones$Valor), beta_portafolio, posiciones$Peso[posiciones$Accion == "PENG"],
              posiciones$Peso[posiciones$Accion == "XOM"], posiciones$Peso[posiciones$Accion == "Efectivo"], var_nvda),
  Despues = c(sum(final$Valor), beta_final, 0, final$Peso[final$Accion == "XOM"],
              final$Peso[final$Accion == "Efectivo"],
              1.645 * vol_nvda$standardDeviation[1] / 100 * final$Valor[final$Accion == "NVDA"]))
comparacion$Cambio <- comparacion$Despues - comparacion$Antes
tabla_html(comparacion, digits = 2, row.names = FALSE, format.args = list(big.mark = ","),
             col.names = c("Indicador", "Antes (6 nov)", "Después (9 nov)", "Cambio"),
             caption = "Comparación del portafolio antes y después de la estrategia")



**Interpretación.**

- **Beta:** la beta del portafolio bajó de 1.04 a **0.94**. Ahora, si el S&P 500 cae 1%, el portafolio cae en promedio 0.94%, es decir, el portafolio pasó a ser ligeramente menos riesgoso que el mercado. La mayor parte de la reducción viene de eliminar PENG, que aportaba 0.106 a la beta, mientras que la compra de XOM solo agrega 0.006.
- **Diversificación:** XOM pasó a pesar 10.6% del portafolio. Como su correlación con las demás acciones es casi nula, el portafolio queda menos expuesto a una caída conjunta de las acciones de tecnología.
- **Límite de 25%:** ninguna posición supera el 25% del portafolio; la mayor es AMZN, con 17.2%.
- **Valor:** la suma de la tabla es de $892,625.98, muy cercana al *Portfolio Value* de *Account Balances* ($892,569.32); la pequeña diferencia se debe a que las dos capturas se tomaron con minutos de diferencia. El rendimiento acumulado mejoró de −11.42% a −10.74%, pero ese cambio se debe sobre todo al alza del mercado del 9 de noviembre (el S&P 500 subió 2.4%), no a la estrategia, cuyo objetivo es reducir el riesgo, no generar una ganancia en pocos días.

# 9. Conclusiones

1. **Regresión simple:** el portafolio tenía una beta de 1.04; después de vender PENG y comprar XOM, bajó a 0.94. Las acciones de tecnología y ciclo económico (PENG, SYNA, NVDA, FCX, CAT) tienen betas mayores a 1.3, mientras que WMT, COST y XOM son defensivas.
2. **Correlación:** WMT, COST y XOM son las que más diversifican, porque casi no se mueven con el resto del portafolio.
3. **Regresión múltiple:** los resultados de RStudio con datos de LSEG coinciden con Bloomberg; con datos de 30 minutos, Walmart se mueve sobre todo con Costco y con el mercado.
4. **Datos de panel:** dentro de cada empresa, más ventas generan más flujo de operación y más inventarios lo reducen; el modelo de efectos fijos es el adecuado según la prueba de Hausman.
5. **Series de tiempo:** los precios de WMT y NVDA tienen raíz unitaria (son caminatas aleatorias) y sus rendimientos son estacionarios. Los pronósticos ARIMA son prácticamente planos con intervalos amplios, lo que confirma que el precio de una acción es muy difícil de predecir.
6. **GARCH:** lo que sí se puede pronosticar es la volatilidad: NVDA es alrededor del doble de volátil que WMT y su volatilidad es persistente, información útil para dimensionar las posiciones del portafolio.
7. **Estrategia:** con base en las betas, las correlaciones y la volatilidad, se vendió PENG (la acción más riesgosa) y se compró XOM (la que más diversifica), con lo que el portafolio quedó con menos riesgo de mercado que el S&P 500.
